In [11]:
import os
import json
from openai import OpenAI
from google.colab import userdata

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from uuid import uuid4 as uuid

In [12]:
# !pip install langchain_openai langchain_chroma

In [13]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage  # Chat history message types
from langchain_core.tools import Tool

In [14]:
# with open('./sample_data/synthetic_tickets.json', 'r') as f:
#     data = json.load(f)

In [15]:
from typing import List

class SupportTicketTools:

  def __init__(self, embedding, llm_model, api_key='openai_IK'):
    self.embedding_model = OpenAIEmbeddings(model=embedding, api_key=userdata.get(api_key))

    self.llm = ChatOpenAI(
      model=llm_model,
      temperature=0,
      api_key=userdata.get(api_key)
    )

    self.conversation_history = []

    self.condense_prompt = ChatPromptTemplate.from_messages([
      ("system", """
        You rewrite follow-up questions into standalone questions.

        Rules:
        - Resolve vague references using chat history.
        - "the ticket", "that ticket", "last ticket", "previous ticket" must be replaced with the exact ticket ID from the most recent relevant assistant message.
        - "that issue", "that resolution", "it", "that" must be replaced with the specific issue/resolution and ticket ID from history.
        - If a ticket ID appears in recent history and the user asks about "the ticket", include that ticket ID.
        - Return only the rewritten question.
        """),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{question}")
    ])

    self.condense_chain = (
        self.condense_prompt | self.llm | StrOutputParser()
    )

    self._load_data()

  def clear_history(self):
    self.conversation_history = []

  def _load_data(self):
    with open('./sample_data/synthetic_tickets.json', 'r') as f:
      self.data = json.load(f)

    documents = []

    for ticket in self.data:
          content = f"""
            Ticket ID: {ticket['ticket_id']}
            Title: {ticket['title']}
            Category: {ticket['category']}
            Priority: {ticket['priority']}
            Date: {ticket['created_date']} to {ticket['resolved_date']}

            Problem Description:
            {ticket['description']}

            Resolution:
            {ticket['resolution']}
          """.strip()

          metadata = {
              'ticket_id': ticket['ticket_id'],
              'title': ticket['title'],
              'category': ticket['category'],
              'priority': ticket['priority'],
              'created_date': ticket['created_date'],
              'resolved_date': ticket['resolved_date'],
          }


          document = Document(page_content=content, metadata=metadata)
          documents.append(document)

    self.vectorstore = Chroma.from_documents(
      documents=documents,
      embedding=self.embedding_model,
      persist_directory="./agent_vectorstore"
    )

    print("Documents loaded in the vector DB 'agent_vectorstore'")

  def search_similar_tickets(self, query: str) -> str:
    _retriever = self.vectorstore.as_retriever(
      search_type="similarity",
      search_kwargs={"k": 3}
    )

    results = _retriever.invoke(query)

    if not results:
      return "No similar tickets found"

    output = self._format_docs(results)

    return output

  def _format_docs(self, docs: List[Document]) -> str:
    output = "Found similar tickets:\n\n"
    for i, doc in enumerate(docs, 1):
      output += f"--- Ticket {i} ---\n"
      output += doc.page_content + "\n\n"

    return output

  def get_ticket_by_id(self, ticket_id: str) -> str:
    for ticket in self.data:
      if ticket['ticket_id'].upper().strip() == ticket_id.upper().strip():
        return f"""
          Ticket ID: {ticket['ticket_id']}
          Title: {ticket['title']}
          Category: {ticket['category']}
          Priority: {ticket['priority']}
          Created_date: {ticket['created_date']}
          Resolved_date: {ticket['resolved_date']}
        """
    return f"Ticket {ticket_id} not found"

  def search_by_category(self, category: str) -> str:
      category = category.strip()
      matching = [t for t in self.data if t['category'].lower() == category.lower()]

      if not matching:
          available = list(set(t['category'] for t in self.data))
          return f"No tickets found in category '{category}'. Available categories: {', '.join(available)}"

      output = f"Found {len(matching)} tickets in category '{category}':\n\n"
      for ticket in matching:
          output += f"• [{ticket['ticket_id']}] {ticket['title']} (Priority: {ticket['priority']})\n"

      return output

  def condense_user_query(self, query: str) -> str:
    if not self.conversation_history:
      return query

    standalone_query = self.condense_chain.invoke({
        "chat_history": self.conversation_history,
        "question": query
    })

    return standalone_query

  def get_tools(self) -> List[Tool]:
      return [
          Tool(
                name="SearchSimilarTickets",
                func=self.search_similar_tickets,
                description="""Use this tool to search for similar support tickets based on a problem description or question.
                Input should be a clear description of the issue or question.
                This is the PRIMARY tool for answering "how to fix" or "similar issue" questions."""
              ),
          Tool(
                name="GetTicketByID",
                func=self.get_ticket_by_id,
                description="""Use this tool to retrieve details of a specific ticket by its ID.
                MANDATORY whenever a ticket ID is known.
                Input should be a ticket ID.
                Use this when the user mentions a specific ticket number."""
              ),
          Tool(
                name="SearchByCategory",
                func=self.search_by_category,
                description="""Use this tool to find all tickets in a specific category.
                Input should be a category name like 'Authentication', 'Database', 'Payment', etc.
                Use this when the user wants to see all issues of a certain type."""
              )
      ]

  def add_to_history(self, user_query: str, assistant_answer: str):
    self.conversation_history.append(
        HumanMessage(content=user_query)
    )
    self.conversation_history.append(
        AIMessage(content=assistant_answer)
    )



In [17]:
llm = ChatOpenAI(
    model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
    temperature=0,
    api_key=userdata.get('openai_IK')
)

tool_manager = SupportTicketTools('text-embedding-3-small', 'gpt-4o-mini')
tools = tool_manager.get_tools()

Documents loaded in the vector DB 'agent_vectorstore'


In [18]:
# Convert tools to OpenAI function format
tool_definitions = []

for tool in tools:
    tool_definitions.append({
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description,
            "parameters": {
                "type": "object",
                "properties": {
                    "input": {
                        "type": "string",
                        "description": "The input to the tool"
                    }
                },
                "required": ["input"]
            }
        }
    })

# Bind tools to LLM
llm_with_tools = llm.bind(tools=tool_definitions)

print(f"✓ Created {len(tools)} tools:")
for tool in tools:
    print(f" - {tool.name}: {tool.description.split('.')[0]}")

✓ Created 3 tools:
 - SearchSimilarTickets: Use this tool to search for similar support tickets based on a problem description or question
 - GetTicketByID: Use this tool to retrieve details of a specific ticket by its ID
 - SearchByCategory: Use this tool to find all tickets in a specific category


In [19]:
def run_agent(query: str, max_iter=3) -> str:

  # Rewrite follow-up query using conversation history
  standalone_query = tool_manager.condense_user_query(query)

  if standalone_query != query:
    print(f"\n📝 Original Query  : {query}")
    print(f"📝 Standalone Query: {standalone_query}")

  messages = [
        SystemMessage(content="""You are an expert support desk assistant that helps troubleshoot technical issues.

        You have access to a database of previous support tickets with their resolutions.
        Use your tools to find relevant information and provide helpful, accurate answers.

        Guidelines:
        - ALWAYS search for similar tickets when asked about troubleshooting or "how to fix" questions
        - Be specific and reference ticket IDs when providing solutions
        - If multiple similar issues exist, mention the most relevant ones
        - Admit when you don't have enough information
        - Be concise but thorough in your responses
        - When appropriate, use multiple tools to gather complete information
        - Before each tool call, provide a short public rationale in content using this exact prefix:
            "Decision: <one sentence explaining why this tool is needed>"

        Remember: Your primary value is retrieving and applying solutions from past tickets!"""),
  ]

  # Add previous conversation
  messages.extend(tool_manager.conversation_history)

  # Use original query for natural conversation
  messages.append(HumanMessage(content=query))

  for i in range(max_iter):

    response = llm_with_tools.invoke(messages)
    messages.append(response)

    # If the model produced no tool calls, treat content as final answer.
    if not response.tool_calls:
        # No more tool calls, return the response
        final_response =  response.content

        # Save only user query + final assistant answer for future turns
        tool_manager.add_to_history(
            user_query=query,
            assistant_answer=final_response
        )

        return final_response

    # Print model-provided public rationale (not hidden chain-of-thought).
    decision_trace = (response.content or "").strip()
    if decision_trace:
        print(f"\n🧭 {decision_trace}")

    list_of_tools = [tool_call["name"] for tool_call in response.tool_calls]
    print(f"Chaining Tools -> {' | '.join([tool_name for tool_name in list_of_tools])}")

    for tool_call in response.tool_calls:
      tool_name = tool_call["name"]
      tool_input = tool_call["args"].get("input", "")

      # Improve tool input only for semantic search
      if tool_name == "SearchSimilarTickets":
          tool_input = standalone_query

      print(f"\n🔧 Calling tool: {tool_name}")
      print(f"   Input: {tool_input}")

      tool_output = None
      for tool in tools:
          if tool.name == tool_name:
              tool_output = tool.func(tool_input)
              break

      if tool_output is None:
          tool_output = f"Error: Tool {tool_name} not found"

      # print(f"   Output: {tool_output[:200]}...")

      messages.append(ToolMessage(
          content=tool_output,
          tool_call_id=tool_call["id"]
      ))

  final_answer = "Maximum iterations reached. Could not complete the task."

  tool_manager.add_to_history(
    user_query=query,
    assistant_answer=final_answer
  )

  return final_answer

print("\n✓ Agent ready!")


✓ Agent ready!


In [22]:
while True:

  if len(tool_manager.conversation_history) == 0:
    print("\nAssistant: How can I help !\n")

  user = input("You: ").strip()

  if user.lower() in ['q', 'exit']:
    tool_manager.clear_history()
    break

  result= run_agent(user)
  print(f"\nAssistant: {result}\n")

You: How do I fix authentication problems after password reset?

📝 Original Query  : How do I fix authentication problems after password reset?
📝 Standalone Query: What steps can I take to fix authentication problems after a password reset?
Chaining Tools -> SearchSimilarTickets

🔧 Calling tool: SearchSimilarTickets
   Input: What steps can I take to fix authentication problems after a password reset?

Assistant: To resolve authentication problems after a password reset, you can refer to the following relevant ticket:

### Ticket ID: TICK-001
- **Title:** Users unable to log in after password reset
- **Problem Description:** Multiple users reported authentication failures with the error message 'Invalid credentials' after performing a password reset. This issue began following a recent security patch deployment.
- **Resolution:**
  - It was identified that the password hash algorithm had been updated, but the session tokens were not invalidated.
  - The solution involved clearing all a

## Gradio UI

In [ ]:
# %pip install gradio

In [29]:
import gradio as gr

def chatbot_interface(message, history):
    message = message.strip()

    if message.lower() in ["clear", "reset"]:
        tool_manager.clear_history()
        return "Conversation history cleared."

    response = run_agent(message)
    return response


gr.ChatInterface(
    fn=chatbot_interface,
    title="Support Ticket Chatbot",
    description="Ask me questions about support tickets! Type 'clear' or 'reset' to start a new conversation.",
    textbox=gr.Textbox(placeholder="Hi, I am your support tooling assistant"),
    theme="soft",
    type="messages"
).launch(share=True, debug=False)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7777bffbdc58de0c14.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



📝 Original Query  : How do I fix authentication problems after password reset?
📝 Standalone Query: To fix authentication problems after a password reset, you can refer to the issue documented in Ticket ID **TICK-001**. Here’s a summary of the problem and its resolution:

### Ticket ID: TICK-001
- **Title:** Users unable to log in after password reset
- **Creation Date:** January 15, 2024
- **Problem Description:** Multiple users reported authentication failures with the error message 'Invalid credentials' after performing a password reset. This issue arose following a recent security patch deployment.
- **Resolution:** 
  - The problem was traced back to an updated password hash algorithm, while session tokens were not invalidated.
  - The solution involved clearing all active sessions and forcing re-authentication for users.
  - Additionally, an automatic session cleanup was implemented upon password changes.

### Recommended Steps:
1. **Clear Active Sessions:** If you have access to

In [ ]:
import gradio as gr

def chatbot_interface(message, history):
    response = run_agent(message)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})
    return "", history

def clear_chat():
    tool_manager.clear_history()
    return []

with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# Support Ticket Chatbot")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(
        placeholder="Hi, I am your support tooling assistant",
        label="Your message"
    )

    send_btn = gr.Button("Send")
    clear_btn = gr.Button("Clear Chat")

    msg.submit(
        fn=chatbot_interface,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    send_btn.click(
        fn=chatbot_interface,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot]
    )

demo.launch()